# connections-rl — publish model cards to the Hugging Face Hub

Regenerates all 8 model cards from the committed evaluation JSON and pushes each
one to its Hub repo as `README.md`.

**No GPU needed.** Settings → Accelerator: None, Internet: **On**.
Add-ons → Secrets → attach a secret named `HF_TOKEN` with **write** scope.
Then Save & Run All. Takes about a minute.

Cards covered:

| Repo | Scale | Stage |
|---|---|---|
| `connections-rl-sft` | 1.5B | SFT |
| `connections-rl-grpo` | 1.5B | GRPO seed 0 |
| `connections-rl-grpo-1.5b-seed1` / `-seed2` | 1.5B | GRPO replicates |
| `connections-rl-sft-7b` | 7B | SFT |
| `connections-rl-grpo-7b` | 7B | GRPO seed 0 |
| `connections-rl-grpo-7b-seed1` / `-seed2` | 7B | GRPO replicates |

Every number in every card is read from `results*/` and `results-analysis/` at
build time. The push step refuses to run if the checked-in cards differ from what
the generator produces, so the Hub cannot drift from the repo's own results.

In [ ]:
# Cell 1 — clone the repo and read the token from Kaggle Secrets
import os, subprocess
from kaggle_secrets import UserSecretsClient

os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

!rm -rf /kaggle/working/connections-rl
!git clone -q https://github.com/jacksonmlukas/connections-rl.git /kaggle/working/connections-rl
%cd /kaggle/working/connections-rl
!pip install -q -U huggingface_hub
!git log --oneline -1

In [ ]:
# Cell 2 — regenerate the cards from the committed eval results
subprocess.run(['python', 'scripts/build_model_cards.py'], check=True)

# fails loudly if generation is not reproducible
subprocess.run(['python', 'scripts/build_model_cards.py', '--check'], check=True)

In [ ]:
# Cell 3 — preview one card before anything is published
from IPython.display import Markdown, display

card = open('hub_cards/connections-rl-grpo-7b.md').read()
print(card[:card.index('---', 4) + 3])          # YAML metadata block
display(Markdown(card[card.index('---', 4) + 3:][:2500] + '\n\n*(truncated preview)*'))

In [ ]:
# Cell 4 — dry run: confirm which repos would be written to
subprocess.run(['python', 'scripts/push_model_cards.py', '--dry-run'], check=True)

# confirm the token works and has the right identity before writing
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
me = api.whoami()
print('authenticated as:', me['name'])
scopes = me.get('auth', {}).get('accessToken', {}).get('role', 'unknown')
print('token role:', scopes)
assert scopes in ('write', 'admin', 'fineGrained'), f'token needs write scope, got {scopes!r}'

In [ ]:
# Cell 5 — push all 8 cards
subprocess.run(['python', 'scripts/push_model_cards.py'], check=True)

In [ ]:
# Cell 6 — verify: re-read each card back off the Hub and diff against local
from huggingface_hub import hf_hub_download
import sys
sys.path.insert(0, 'scripts')
from push_model_cards import REPOS

user = api.whoami()['name']
bad = []
for name in REPOS:
    local = open(f'hub_cards/{name}.md').read()
    try:
        remote = open(hf_hub_download(f'{user}/{name}', 'README.md',
                                      token=os.environ['HF_TOKEN'],
                                      force_download=True)).read()
    except Exception as exc:
        print(f'FAIL  {name}: {exc}'); bad.append(name); continue
    ok = remote.strip() == local.strip()
    print(('OK    ' if ok else 'DIFF  ') + f'https://huggingface.co/{user}/{name}')
    if not ok:
        bad.append(name)

print()
print('all cards verified live on the Hub' if not bad else f'PROBLEMS: {bad}')